# v31 unified one-run v3 positional alignment — candidate-pool rescue + artifact-safe selector

Gộp 4 notebook v31 vào 1 notebook duy nhất để tránh lỗi mất `/kaggle/working` giữa notebook.

**Locked baseline:** `v30.1 / v30 unified`, macro-F1 = `0.5934206145879246`.

Notebook này chạy đủ:

1. `v31_a` — candidate-pool signal analysis, đo precision của trigger R3 trước khi tin rule.
2. `v31_standard` — conservative candidate-pool rescue.
3. `v31_b` — contradiction ablation, chỉ để phân tích.
4. `v31_full` — operational selector, chỉ select nếu summary và saved preds thật đều verify.

Nguyên tắc bắt buộc:

- Không train model.
- Không đụng MC.
- Không đụng 3 flips đã khóa của v30.1: `{71, 109, 125}`.
- Không dùng grid-search.
- Không select nếu chỉ summary tốt nhưng preds file hỏng.
- Nếu bất kỳ artifact nào sai, rollback về `v30_1_full_preds.json`.



In [ ]:
# ============================================================
# v31 UNIFIED CORE
# ============================================================
# Self-contained one-run notebook.
# Required inputs:
#   v27_standard_preds.json
#   v30_1_full_preds.json
#   v23_val_candidates.json
# Optional:
#   v30_1_full_summary.json
# Outputs:
#   v31_a_preds.json / v31_a_summary.json
#   v31_standard_preds.json / v31_standard_summary.json
#   v31_b_preds.json / v31_b_summary.json
#   v31_full_preds.json / v31_full_summary.json
#   v31_unified_report.json
# ============================================================

import json, re, glob, os, random, math, traceback
from collections import Counter, defaultdict

LABELS = ['A','B','C','D','Yes','No','Unknown']
YNU = ['Yes','No','Unknown']
V27_MACRO  = 0.5833102471757934
V301_MACRO = 0.5934206145879246
V301_DIFFS = {71, 109, 125}
MIN_R3_PRECISION = 0.70
STD_GAIN_GATE = 0.0100
BOOT_PPOS_GATE = 0.80

WORKDIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/mnt/data'
os.makedirs(WORKDIR, exist_ok=True)
print('WORKDIR =', WORKDIR)

SEARCH_ROOTS = [WORKDIR, '/kaggle/input', '/mnt/data']
EXPLICIT_PATHS = {
    'v27_standard_preds.json': [
        '/kaggle/input/datasets/yixuanisthebest/v2333333/v27_standard_preds.json',
        '/mnt/data/v27_standard_preds.json',
    ],
    'v30_1_full_preds.json': [
        '/kaggle/input/datasets/yixuanisthebest/v2333333/v30_1_full_preds.json',
        '/mnt/data/v30_1_full_preds.json',
    ],
    'v30_1_full_summary.json': [
        '/kaggle/input/datasets/yixuanisthebest/v2333333/v30_1_full_summary.json',
        '/mnt/data/v30_1_full_summary.json',
    ],
    'v23_val_candidates.json': [
        '/kaggle/input/datasets/yixuanisthebest/v2333333/v23_val_candidates.json',
        '/mnt/data/v23_val_candidates.json',
    ],
}

def find_file(fname, required=True, extra=()):
    tried = []
    for p in list(extra) + EXPLICIT_PATHS.get(fname, []):
        tried.append(p)
        if p and os.path.exists(p):
            return p
    for root in SEARCH_ROOTS:
        if not os.path.exists(root):
            continue
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        hits = [h for h in hits if os.path.isfile(h)]
        tried.extend(hits[:5])
        if hits:
            # Prefer non-parenthesized filenames in /kaggle/working if present.
            hits = sorted(hits, key=lambda x: (('(' in os.path.basename(x)), len(x), x))
            return hits[0]
    if required:
        raise FileNotFoundError(f'FATAL: {fname} not found. Tried roots={SEARCH_ROOTS}; samples={tried[:10]}')
    return None

def load_json(fname, required=True, extra=()):
    p = find_file(fname, required=required, extra=extra)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def dump_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)

def get_pred(r):
    for k in ('pred','prediction','answer','final_answer'):
        if k in r:
            return r[k]
    raise KeyError(f'No prediction key in row keys={list(r.keys())}')

def set_pred(r, value):
    # Preserve existing schema; most project files use 'pred'.
    if 'pred' in r:
        r['pred'] = value
    elif 'prediction' in r:
        r['prediction'] = value
    else:
        r['pred'] = value

def metrics(rows):
    tp, fp, fn = Counter(), Counter(), Counter()
    gold_count, pred_count = Counter(), Counter()
    for r in rows:
        g, p = r['gold'], get_pred(r)
        gold_count[g] += 1
        pred_count[p] += 1
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    per = {}
    for l in LABELS:
        pr = tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0.0
        rc = tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        per[l] = dict(precision=pr, recall=rc, f1=f1, support=gold_count[l], pred_count=pred_count[l])
    total = sum(gold_count[l] for l in LABELS)
    return dict(
        n=len(rows),
        acc=sum(r['gold']==get_pred(r) for r in rows)/len(rows),
        macro_f1=sum(per[l]['f1'] for l in LABELS)/len(LABELS),
        weighted_f1=sum(per[l]['f1']*per[l]['support'] for l in LABELS)/total,
        mc_macro=sum(per[l]['f1'] for l in 'ABCD')/4,
        ynu_macro=sum(per[l]['f1'] for l in YNU)/3,
        per_label=per,
        gold_count=dict(gold_count),
        pred_count=dict(pred_count),
    )

def compact_metrics(m):
    return {k:m[k] for k in ('n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro')}

def show(m, name):
    print(f'--- {name} ---')
    print(f"acc={m['acc']:.4f} macro-F1={m['macro_f1']:.6f} weighted-F1={m['weighted_f1']:.4f} MC_macro={m['mc_macro']:.4f} YNU_macro={m['ynu_macro']:.4f}")
    for l in LABELS:
        v = m['per_label'][l]
        flag = ' COLLAPSE' if v['pred_count']==0 and v['support']>0 else (' LOW_F1' if v['f1']<0.35 else '')
        print(f"  {l:8s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1']:.3f} gold={v['support']} pred={v['pred_count']}{flag}")

def diffs_between(a_rows, b_rows):
    assert len(a_rows) == len(b_rows), f'length mismatch: {len(a_rows)} vs {len(b_rows)}'
    return sorted(a['idx'] for a, b in zip(a_rows, b_rows) if get_pred(a) != get_pred(b))

def save_and_verify(preds, summary, tag, v27_rows, expected_diffs=None, min_macro=None, mc_ref=None):
    """Verify ACTUAL SAVED FILE, not in-memory objects."""
    pp = os.path.join(WORKDIR, f'{tag}_preds.json')
    sp = os.path.join(WORKDIR, f'{tag}_summary.json')
    dump_json(preds, pp)
    dump_json(summary, sp)

    re_p, _ = load_json(os.path.basename(pp), extra=[pp])
    re_s, _ = load_json(os.path.basename(sp), extra=[sp])
    re_m = metrics(re_p)
    summ_m = re_s.get('metrics') or re_s.get('new_metrics') or re_s.get('selected_metrics')
    assert summ_m is not None, f'{tag}: summary has no metrics/new_metrics/selected_metrics'
    assert abs(re_m['macro_f1'] - summ_m['macro_f1']) < 1e-9, f'{tag}: saved preds macro {re_m["macro_f1"]} != summary {summ_m["macro_f1"]}'
    if min_macro is not None:
        assert re_m['macro_f1'] >= min_macro - 1e-12, f'{tag}: macro {re_m["macro_f1"]} < min {min_macro}'
    if mc_ref is not None:
        assert abs(re_m['mc_macro'] - mc_ref) < 1e-12, f'{tag}: MC changed {re_m["mc_macro"]} != {mc_ref}'
    diffs = diffs_between(v27_rows, re_p)
    if expected_diffs is not None:
        assert set(diffs) == set(expected_diffs), f'{tag}: saved diffs {diffs} != expected {sorted(expected_diffs)}'
    print(f'SAVED+VERIFIED {tag}: {pp} / {sp} | macro={re_m["macro_f1"]:.10f} | diffs_vs_v27={diffs}')
    return pp, sp, re_m, diffs


In [ ]:
# ============================================================
# Load + hard-verify locked baseline v30.1
# ============================================================

v27_rows, v27_path = load_json('v27_standard_preds.json')
v30_rows, v30_path = load_json('v30_1_full_preds.json')
v30_summary, v30_summary_path = load_json('v30_1_full_summary.json', required=False)

m27 = metrics(v27_rows)
mb = metrics(v30_rows)
base_diffs = diffs_between(v27_rows, v30_rows)

print('Loaded v27:', v27_path)
print('Loaded v30.1:', v30_path)
print('Loaded v30.1 summary:', v30_summary_path)
show(m27, 'v27 baseline')
show(mb, 'v30.1 locked baseline')

assert abs(m27['macro_f1'] - V27_MACRO) < 1e-9, f'v27 macro mismatch: {m27["macro_f1"]}'
assert abs(mb['macro_f1'] - V301_MACRO) < 1e-9, f'v30.1 macro mismatch: {mb["macro_f1"]}'
assert set(base_diffs) == V301_DIFFS, f'v30.1 diffs {base_diffs} != {sorted(V301_DIFFS)}'
assert abs(mb['mc_macro'] - m27['mc_macro']) < 1e-12, 'MC freeze violated in v30.1 baseline'

print('LOCKED BASELINE VERIFIED:', dict(macro=mb['macro_f1'], diffs=base_diffs, mc_macro=mb['mc_macro']))


In [ ]:
# ============================================================
# Candidate pool loading + schema-defensive parser
# v3 fix: align candidate records by global LIST POSITION first.
# Important: in v23_val_candidates.json, q_idx is only 0/1 inside each sample, not global 0..155.
# sample_id is sample-level and also not a unique question index.
# The candidate file is a 156-row list in validation question order, so enumerate position is the safest primary index.
# Exact question matching is kept as a sanity check / fallback.
# ============================================================

raw_candidates, cand_path = load_json('v23_val_candidates.json')
print('Loaded candidates:', cand_path, 'type=', type(raw_candidates).__name__)

FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)

def normalize_label(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    up = s.upper()
    if up in ['A','B','C','D']:
        return up
    t = s.title()
    if t in ['Yes','No','Unknown']:
        return t
    return None

def cand_answer(c):
    if isinstance(c, str):
        m = FINAL_RE.findall(c)
        return normalize_label(m[-1]) if m else None
    if isinstance(c, dict):
        for k in ('answer','final_answer','final','pred','prediction','ans','label'):
            if k in c and c[k] is not None:
                lab = normalize_label(c[k])
                if lab:
                    return lab
        for k in ('text','raw','output','explanation','completion','response','generation','content'):
            if k in c and c[k]:
                m = FINAL_RE.findall(str(c[k]))
                if m:
                    return normalize_label(m[-1])
    return None

def cand_text(c):
    if isinstance(c, str):
        return c
    if isinstance(c, dict):
        parts = []
        for k in ('text','raw','output','explanation','completion','response','generation','content'):
            if k in c and c[k]:
                parts.append(str(c[k]))
        if parts:
            return '\n'.join(parts)
        return json.dumps(c, ensure_ascii=False)[:4000]
    return str(c)

def candidate_records(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for k in ('data','samples','records','items'):
            if isinstance(raw.get(k), list):
                return raw[k]
        recs = []
        for k, v in raw.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault('_dict_key_idx', int(k) if str(k).isdigit() else k)
                recs.append(vv)
            elif isinstance(v, list):
                recs.append({'_dict_key_idx': int(k) if str(k).isdigit() else k, 'candidates': v})
        return recs
    raise AssertionError(f'Unexpected candidate file type: {type(raw).__name__}')

def build_pool(raw, preds):
    recs = candidate_records(raw)
    assert recs, 'candidate file empty'
    print('candidate records:', len(recs), 'first type:', type(recs[0]).__name__)
    if isinstance(recs[0], dict):
        print('candidate record keys:', sorted(recs[0].keys())[:30])

    def rec_cands(rec):
        if isinstance(rec, list):
            return rec
        if not isinstance(rec, dict):
            return None
        for k in ('candidates','cands','outputs','generations','samples','completions','responses','predictions'):
            if isinstance(rec.get(k), list):
                return rec[k]
        if cand_answer(rec):
            return [rec]
        return None

    norm = lambda q: re.sub(r'\s+',' ', str(q).strip().lower())[:500]
    byq = {norm(r.get('question','')): int(r['idx']) for r in preds if r.get('question') and 'idx' in r}

    pool = {}
    skipped = 0
    question_match = 0
    positional_used = 0
    question_mismatch_examples = []

    # The known v23_val_candidates.json is list-aligned to validation rows.
    # q_idx is repeated 0/1 and sample_id is not unique per question, so NEVER use them as primary global ids.
    for pos, rec in enumerate(recs):
        cl = rec_cands(rec)
        if cl is None:
            skipped += 1
            continue

        i = None
        q_i = None
        if isinstance(rec, dict):
            for k in ('question','query','prompt'):
                if k in rec:
                    q_i = byq.get(norm(rec[k]))
                    if q_i is not None:
                        break

        if len(recs) == len(preds):
            i = pos
            positional_used += 1
            if q_i is not None:
                if q_i == pos:
                    question_match += 1
                elif len(question_mismatch_examples) < 5:
                    question_mismatch_examples.append({'pos': pos, 'question_match_idx': q_i, 'candidate_question': str(rec.get('question',''))[:120], 'pred_question_at_pos': str(preds[pos].get('question',''))[:120]})
        elif q_i is not None:
            i = q_i
            question_match += 1
        elif isinstance(rec, dict) and '_dict_key_idx' in rec:
            i = rec['_dict_key_idx']
        else:
            skipped += 1
            continue

        try:
            i = int(i)
        except Exception:
            pass
        if i in pool:
            raise AssertionError(f'Duplicate global candidate index {i}. Alignment is unsafe; STOP.')
        pool[i] = [dict(ans=cand_answer(c), text=cand_text(c), raw_type=type(c).__name__) for c in cl]

    ynu_n = len([r for r in preds if not r.get('is_mc')])
    ynu_cov = len([i for i, r in enumerate(preds) if (not r.get('is_mc')) and i in pool])
    print(f'pool built for {len(pool)}/{len(preds)} rows; YNU rows={ynu_n}; YNU covered={ynu_cov}; skipped={skipped}; sample sizes={Counter(len(v) for v in pool.values())}')
    print(f'alignment: positional_used={positional_used}, exact_question_matches={question_match}')
    if question_mismatch_examples:
        print('WARNING: positional alignment used, but some question exact matches point elsewhere. Examples:')
        for ex in question_mismatch_examples:
            print(ex)

    # Hard checks. If these fail, do not trust v31.
    assert len(pool) >= 0.95 * len(preds), 'pool coverage too low; alignment likely broken. STOP.'
    assert ynu_cov >= 0.95 * ynu_n, 'YNU pool coverage too low; alignment likely broken. STOP.'
    if len(recs) == len(preds):
        # At least most questions should match by text. If not, order may still be correct, but be conservative.
        assert question_match >= 0.80 * len(preds), f'question text match too low ({question_match}/{len(preds)}); positional alignment unsafe. STOP.'
    return pool

pool = build_pool(raw_candidates, v30_rows)



In [ ]:
# ============================================================
# Signals + rescue rules
# ============================================================

CONTRA_NEG = re.compile(r'(cannot conclude|does not follow|not necessarily|insufficient|cannot determine|not supported|no premise|fallacy|unknown|cannot be inferred|not guaranteed)', re.I)
AFFIRM = re.compile(r'\b(thus|therefore|hence|so|consequently)\b[^.]{0,120}\b(is|are|does|do|will|can|qualifies|receives|completes?|meets?|holds|follows|true)\b', re.I)
UNIV_TRAP = re.compile(r'\b(all|every|each|exists|there exists|some|at least one|forall|for all|∃|∀)\b', re.I)

def body_final_contradiction(text):
    m = FINAL_RE.search(text or '')
    if not m:
        return False
    fin = normalize_label(m.group(1))
    body = text[:m.start()]
    aff = bool(AFFIRM.search(body))
    neg = bool(CONTRA_NEG.search(body))
    return (fin == 'No' and aff and not neg) or (fin == 'Yes' and neg and not aff)

def citations_valid(text, n_premises=None):
    if not text:
        return False
    m = re.search(r'Supporting Premises\s*[:\-]?\s*\[([^\]]*)\]', text, re.I)
    if not m:
        return False
    ids = re.findall(r'\d+', m.group(1))
    if not ids:
        return False
    if n_premises:
        return all(0 < int(i) <= n_premises for i in ids)
    return True

def case_signals(r, pool):
    cl = pool.get(r['idx'], [])
    ynu_ans = [c['ans'] for c in cl if c.get('ans') in YNU]
    n = len(ynu_ans)
    vc = Counter(ynu_ans)
    yes_share = vc['Yes']/n if n else 0.0
    no_share = vc['No']/n if n else 0.0
    unk_share = vc['Unknown']/n if n else 0.0
    yes_cited = any(c.get('ans')=='Yes' and citations_valid(c.get('text','')) for c in cl)
    no_cited = any(c.get('ans')=='No' and citations_valid(c.get('text','')) for c in cl)
    contra_no = sum(1 for c in cl if c.get('ans')=='No' and body_final_contradiction(c.get('text','')))
    contra_yes = sum(1 for c in cl if c.get('ans')=='Yes' and body_final_contradiction(c.get('text','')))
    trap = bool(UNIV_TRAP.search(r.get('question','')))
    return dict(
        n_cands=n,
        yes_share=yes_share,
        no_share=no_share,
        unknown_share=unk_share,
        yes_cited=yes_cited,
        no_cited=no_cited,
        n_no_contradictory=contra_no,
        n_yes_contradictory=contra_yes,
        quant_trap=trap,
        vote=dict(vc),
    )

def r3_trigger(row, pool, variant='standard'):
    if row.get('is_mc') or row['idx'] in V301_DIFFS or get_pred(row) != 'No':
        return False, case_signals(row, pool), 'not eligible'
    s = case_signals(row, pool)
    if s['n_cands'] < 3:
        return False, s, 'n_cands < 3'
    if variant == 'standard':
        th_plain, th_trap, req_contra = 0.60, 0.80, False
    elif variant == 'b':
        th_plain, th_trap, req_contra = 0.50, 0.70, True
    else:
        raise ValueError(variant)
    th = th_trap if s['quant_trap'] else th_plain
    ok = s['yes_share'] >= th and s['yes_cited']
    reason = f"yes_share={s['yes_share']:.2f} th={th:.2f} trap={s['quant_trap']} yes_cited={s['yes_cited']}"
    if req_contra:
        ok = ok and s['n_no_contradictory'] >= 1
        reason += f" no_contra={s['n_no_contradictory']}"
    return ok, s, reason

def apply_rescue(base_rows, pool, variant='standard', allow=True):
    out, flips = [], []
    for r in base_rows:
        nr = dict(r)
        if allow and (not r.get('is_mc')) and r['idx'] not in V301_DIFFS:
            pred = get_pred(r)
            if pred == 'No':
                ok, s, reason = r3_trigger(r, pool, variant=variant)
                if ok:
                    set_pred(nr, 'Yes')
                    nr['repair'] = f'v31_R3_{variant}: {reason}'
            elif pred == 'UNPARSEABLE':
                s = case_signals(r, pool)
                if s['n_cands'] >= 1 and s['vote']:
                    top = max(s['vote'], key=s['vote'].get)
                    set_pred(nr, top)
                    nr['repair'] = f'v31_R5_pool_majority: {top} vote={s["vote"]}'
            if get_pred(nr) != get_pred(r):
                flips.append((r['idx'], get_pred(r), get_pred(nr), r['gold'], nr.get('repair','')))
        out.append(nr)
    return out, flips

def bootstrap_gain(base, new, n_boot=400, seed=0):
    def mac(sub):
        tp,fp,fn=Counter(),Counter(),Counter()
        for r in sub:
            g,p=r['gold'],get_pred(r)
            if g==p: tp[g]+=1
            else: fp[p]+=1; fn[g]+=1
        f1s=[]
        for l in LABELS:
            pr=tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0.0
            rc=tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0.0
            f1s.append(2*pr*rc/(pr+rc) if pr+rc else 0.0)
        return sum(f1s)/len(f1s)
    rnd=random.Random(seed)
    n=len(base)
    gains=[]
    for _ in range(n_boot):
        ix=[rnd.randrange(n) for _ in range(n)]
        gains.append(mac([new[i] for i in ix]) - mac([base[i] for i in ix]))
    gains.sort()
    return dict(
        mean=sum(gains)/len(gains),
        ci_lo=gains[int(0.05*len(gains))],
        ci_hi=gains[int(0.95*len(gains))],
        p_positive=sum(g > 0 for g in gains)/len(gains),
    )

def decide(base_m, new_m, boot, name):
    gain = new_m['macro_f1'] - base_m['macro_f1']
    unk_drop = base_m['per_label']['Unknown']['f1'] - new_m['per_label']['Unknown']['f1']
    no_drop  = base_m['per_label']['No']['f1'] - new_m['per_label']['No']['f1']
    yes_drop = base_m['per_label']['Yes']['f1'] - new_m['per_label']['Yes']['f1']
    collapse = any(v['pred_count']==0 and v['support']>0 for v in new_m['per_label'].values())
    mc_ok = abs(new_m['mc_macro'] - base_m['mc_macro']) < 1e-12
    ok = (
        gain > STD_GAIN_GATE and
        boot['p_positive'] >= BOOT_PPOS_GATE and
        unk_drop <= 0.0 and
        no_drop <= 0.03 and
        yes_drop <= 0.0 and
        not collapse and
        mc_ok
    )
    status = name if ok else 'ROLLBACK_v30_1'
    gates = dict(
        gain=gain,
        min_gain=STD_GAIN_GATE,
        bootstrap_p_positive=boot['p_positive'],
        min_bootstrap_p_positive=BOOT_PPOS_GATE,
        unknown_f1_drop=unk_drop,
        no_f1_drop=no_drop,
        yes_f1_drop=yes_drop,
        collapse=collapse,
        mc_frozen=mc_ok,
        selected=status,
    )
    print(f'DECISION {name}: {status} | gain={gain:+.6f}, P+={boot["p_positive"]:.2f}, unk_drop={unk_drop:+.3f}, no_drop={no_drop:+.3f}, yes_drop={yes_drop:+.3f}, collapse={collapse}, mc_frozen={mc_ok}')
    return status, gates


In [ ]:
# ============================================================
# v31_a — candidate-pool analysis FIRST
# ============================================================
# This section measures trigger precision using VAL labels.
# If R3 precision < 0.70, standard is automatically blocked.

analysis_rows = []
for r in v30_rows:
    if r.get('is_mc'):
        continue
    s = case_signals(r, pool)
    trig, _, reason = r3_trigger(r, pool, variant='standard')
    analysis_rows.append(dict(
        idx=r['idx'],
        gold=r['gold'],
        pred=get_pred(r),
        correct=(r['gold'] == get_pred(r)),
        r3_trigger=trig,
        r3_reason=reason,
        **s,
    ))

ynu_with_pool = [x for x in analysis_rows if x['n_cands'] > 0]
def pool_has_gold(x):
    return x['vote'].get(x['gold'], 0) > 0
pool_oracle = sum(pool_has_gold(x) for x in ynu_with_pool)

r3_rows = [x for x in analysis_rows if x['r3_trigger']]
r3_precision = (sum(x['gold']=='Yes' for x in r3_rows) / len(r3_rows)) if r3_rows else 0.0
r3_gold_counts = Counter(x['gold'] for x in r3_rows)

print('YNU with pool:', len(ynu_with_pool), '/', len(analysis_rows))
print('Pool oracle coverage:', pool_oracle, '/', len(ynu_with_pool), '=', (pool_oracle/len(ynu_with_pool) if ynu_with_pool else 0.0))
print('R3 trigger count:', len(r3_rows), 'precision gold=Yes:', r3_precision, 'gold_counts:', dict(r3_gold_counts))
print('\nTop R3 trigger rows:')
for x in r3_rows[:50]:
    print(f"  idx={x['idx']:3d} pred={x['pred']} gold={x['gold']} yes_share={x['yes_share']:.2f} no_share={x['no_share']:.2f} trap={x['quant_trap']} yes_cited={x['yes_cited']} vote={x['vote']}")

ALLOW_STANDARD = (len(r3_rows) > 0 and r3_precision >= MIN_R3_PRECISION)
print('ALLOW_STANDARD =', ALLOW_STANDARD)

v31_a_summary = dict(
    version='v31_a_unified_candidate_pool_analysis',
    purpose='Analysis only; no final selection. Measures R3 precision before running standard.',
    baseline_metrics=compact_metrics(mb),
    baseline_per_label=mb['per_label'],
    pool_coverage=dict(ynu_with_pool=len(ynu_with_pool), ynu_total=len(analysis_rows), pool_oracle=pool_oracle,
                       pool_oracle_rate=pool_oracle/len(ynu_with_pool) if ynu_with_pool else 0.0),
    r3_precision_gate=dict(min_precision=MIN_R3_PRECISION, trigger_count=len(r3_rows), precision_gold_yes=r3_precision,
                           gold_counts=dict(r3_gold_counts), allow_standard=ALLOW_STANDARD),
    r3_rows=r3_rows,
    analysis_rows=analysis_rows,
)
# v31_a_preds is intentionally a copy of v30.1 baseline.
save_and_verify(v30_rows, {**v31_a_summary, 'metrics': compact_metrics(mb)}, 'v31_a', v27_rows, expected_diffs=V301_DIFFS, min_macro=V301_MACRO, mc_ref=m27['mc_macro'])


In [ ]:
# ============================================================
# v31_standard — conservative candidate-pool rescue
# ============================================================
# R3: pred No -> Yes when yes_share >= 0.60 (>=0.80 if quantifier trap) and yes_cited.
# R5: UNPARSEABLE -> pool majority.
# Does not touch MC / Unknown / locked v30 flips {71,109,125}.
# If v31_a blocks standard, this section saves rollback baseline with clear reason.

std_new, std_flips = apply_rescue(v30_rows, pool, variant='standard', allow=ALLOW_STANDARD)
std_m = metrics(std_new)
std_boot = bootstrap_gain(v30_rows, std_new)
std_status, std_gates = decide(mb, std_m, std_boot, 'v31_standard') if ALLOW_STANDARD else ('ROLLBACK_v30_1', dict(blocked_by_v31_a=True, reason=f'R3 precision {r3_precision:.3f} < {MIN_R3_PRECISION} or no triggers'))

print('\nv31_standard flips:', len(std_flips), 'correct_on_val:', sum(1 for f in std_flips if f[2] == f[3]))
for f in std_flips:
    print('  idx=%3d %s->%s gold=%s | %s' % f)
show(std_m, 'v31_standard candidate result')
print('std_status =', std_status)

std_selected = std_new if std_status == 'v31_standard' else v30_rows
std_selected_m = std_m if std_status == 'v31_standard' else mb
std_expected_diffs = sorted(V301_DIFFS | {f[0] for f in std_flips}) if std_status == 'v31_standard' else sorted(V301_DIFFS)

v31_standard_summary = dict(
    version='v31_standard_unified_candidate_pool_rescue',
    selection_status=std_status,
    reason='Selected v31_standard after gates' if std_status == 'v31_standard' else 'Rollback to v30.1 because gates failed or v31_a blocked standard',
    baseline_v30_1=compact_metrics(mb),
    candidate_metrics=compact_metrics(std_m),
    metrics=compact_metrics(std_selected_m),
    per_label=std_selected_m['per_label'],
    pred_count=std_selected_m['pred_count'],
    n_flips=len(std_flips),
    flips=[list(f) for f in std_flips],
    expected_diffs_vs_v27=std_expected_diffs,
    thresholds=dict(plain=0.60, trap=0.80, min_r3_precision=MIN_R3_PRECISION, declared='a priori, not grid-searched'),
    bootstrap=std_boot,
    gates=std_gates,
    v31_a_gate=v31_a_summary['r3_precision_gate'],
)
save_and_verify(std_selected, v31_standard_summary, 'v31_standard', v27_rows, expected_diffs=std_expected_diffs, min_macro=V301_MACRO, mc_ref=m27['mc_macro'])


In [ ]:
# ============================================================
# v31_b — contradiction variant / ablation only
# ============================================================
# More experimental: looser yes_share but requires contradictory No-candidate.
# It is saved for analysis but must not be used as final unless v31_full explicitly selects it.
# Current v31_full only considers v31_standard, not v31_b.

b_new, b_flips = apply_rescue(v30_rows, pool, variant='b', allow=True)
b_m = metrics(b_new)
b_boot = bootstrap_gain(v30_rows, b_new)
b_status, b_gates = decide(mb, b_m, b_boot, 'v31_b')

print('\nv31_b flips:', len(b_flips), 'correct_on_val:', sum(1 for f in b_flips if f[2] == f[3]))
for f in b_flips:
    print('  idx=%3d %s->%s gold=%s | %s' % f)
show(b_m, 'v31_b candidate result')

# Save selected if gates pass, otherwise rollback baseline. Summary retains candidate_metrics/flips.
b_selected = b_new if b_status == 'v31_b' else v30_rows
b_selected_m = b_m if b_status == 'v31_b' else mb
b_expected_diffs = sorted(V301_DIFFS | {f[0] for f in b_flips}) if b_status == 'v31_b' else sorted(V301_DIFFS)

v31_b_summary = dict(
    version='v31_b_unified_contradiction_variant',
    purpose='Ablation only; v31_full does not select this by default.',
    selection_status=b_status,
    baseline_v30_1=compact_metrics(mb),
    candidate_metrics=compact_metrics(b_m),
    metrics=compact_metrics(b_selected_m),
    per_label=b_selected_m['per_label'],
    pred_count=b_selected_m['pred_count'],
    n_flips=len(b_flips),
    flips=[list(f) for f in b_flips],
    expected_diffs_vs_v27=b_expected_diffs,
    thresholds=dict(plain=0.50, trap=0.70, requires_no_candidate_body_final_contradiction=True),
    bootstrap=b_boot,
    gates=b_gates,
)
save_and_verify(b_selected, v31_b_summary, 'v31_b', v27_rows, expected_diffs=b_expected_diffs, min_macro=V301_MACRO if b_status!='v31_b' else None, mc_ref=m27['mc_macro'])


In [ ]:
# ============================================================
# v31_full — operational selector
# ============================================================
# Select v31_standard only if:
#   - v31_standard_summary says selection_status == v31_standard
#   - actual saved v31_standard_preds recomputes to summary metrics
#   - actual saved preds beat locked baseline
#   - MC frozen
#   - diffs match summary expected_diffs
# Else rollback to v30.1.

full_source = 'ROLLBACK_v30_1'
full_reason = 'Default rollback'
full_rows = v30_rows
full_m = mb

try:
    std_preds, std_preds_path = load_json('v31_standard_preds.json', extra=[os.path.join(WORKDIR, 'v31_standard_preds.json')])
    std_summary, std_summary_path = load_json('v31_standard_summary.json', extra=[os.path.join(WORKDIR, 'v31_standard_summary.json')])
    std_saved_m = metrics(std_preds)
    std_expected = set(std_summary.get('expected_diffs_vs_v27', []))
    std_actual_diffs = set(diffs_between(v27_rows, std_preds))
    ok = (
        std_summary.get('selection_status') == 'v31_standard' and
        abs(std_saved_m['macro_f1'] - std_summary['metrics']['macro_f1']) < 1e-9 and
        std_saved_m['macro_f1'] > V301_MACRO and
        abs(std_saved_m['mc_macro'] - mb['mc_macro']) < 1e-12 and
        std_actual_diffs == std_expected
    )
    if ok:
        full_rows = std_preds
        full_m = std_saved_m
        full_source = 'v31_standard_saved_preds_verified'
        full_reason = f'Selected v31_standard from {std_preds_path}; summary and saved preds verified.'
    else:
        full_reason = (
            'Rollback: v31_standard did not pass full verification. '
            f"selection_status={std_summary.get('selection_status')}; "
            f"saved_macro={std_saved_m['macro_f1']:.10f}; summary_macro={std_summary.get('metrics',{}).get('macro_f1')}; "
            f"actual_diffs={sorted(std_actual_diffs)} expected_diffs={sorted(std_expected)}"
        )
except Exception as e:
    full_reason = 'Rollback: exception while verifying v31_standard artifact: ' + repr(e)
    print(full_reason)

show(full_m, 'v31_full FINAL')
print('full_source =', full_source)
print('full_reason =', full_reason)

full_expected_diffs = diffs_between(v27_rows, full_rows)
v31_full_summary = dict(
    version='v31_full_unified_select_or_rollback',
    selection_status='SELECT_V31' if full_source.startswith('v31_standard') else 'ROLLBACK_TO_V30_1',
    selected_source=full_source,
    reason=full_reason,
    metrics=compact_metrics(full_m),
    per_label=full_m['per_label'],
    pred_count=full_m['pred_count'],
    base_metrics=compact_metrics(mb),
    delta_vs_v30_1={k: full_m[k]-mb[k] for k in ('acc','macro_f1','weighted_f1','mc_macro','ynu_macro')},
    diffs_vs_v27=full_expected_diffs,
    diffs_vs_v30_1=diffs_between(v30_rows, full_rows),
    v31_a_gate=v31_a_summary['r3_precision_gate'],
    standard_status=v31_standard_summary['selection_status'],
    standard_candidate_metrics=v31_standard_summary['candidate_metrics'],
    b_status=v31_b_summary['selection_status'],
    b_candidate_metrics=v31_b_summary['candidate_metrics'],
)

min_macro = V301_MACRO if full_source == 'ROLLBACK_v30_1' else V301_MACRO + 1e-12
save_and_verify(full_rows, v31_full_summary, 'v31_full', v27_rows, expected_diffs=full_expected_diffs, min_macro=min_macro, mc_ref=m27['mc_macro'])

# Convenience alias if selected; otherwise it still points to final rollback file.
save_and_verify(full_rows, {**v31_full_summary, 'version':'v31_1_full_unified_alias'}, 'v31_1_full', v27_rows, expected_diffs=full_expected_diffs, min_macro=min_macro, mc_ref=m27['mc_macro'])


In [ ]:
# ============================================================
# Final report + next-direction notes
# ============================================================

v31_unified_report = dict(
    version='v31_unified_one_run_report',
    best_before='v30.1',
    locked_v30_1_macro=V301_MACRO,
    final_selection=v31_full_summary['selection_status'],
    final_source=v31_full_summary['selected_source'],
    final_metrics=v31_full_summary['metrics'],
    final_delta_vs_v30_1=v31_full_summary['delta_vs_v30_1'],
    artifacts={
        'v31_a_summary':'v31_a_summary.json',
        'v31_standard_summary':'v31_standard_summary.json',
        'v31_b_summary':'v31_b_summary.json',
        'v31_full_summary':'v31_full_summary.json',
        'v31_1_full_summary':'v31_1_full_summary.json',
    },
    warnings=[
        'v31_b is ablation/prototype; do not feed v31_b_preds into best selector unless explicitly verified.',
        'Future versions must verify saved preds, not only summary metrics.',
        'If v31 rolls back, candidate pool signals were insufficient under high-precision gates; consider new candidate generation or external symbolic candidate scoring.',
    ],
    questions_for_claude=[
        'Is the R3 precision gate leaking validation labels too directly? Should it be reported only as diagnostic while standard thresholds remain fixed?',
        'Should v31_full require bootstrap p_positive >= 0.8, or is exact macro gain enough for this small validation set?',
        'Are yes_share thresholds 0.60/0.80 too permissive given v30 fixed Yes->No overclaim cases by flipping Yes to No?',
        'Can candidate citation_valid be gamed by hallucinated premise IDs? Should we validate IDs against actual premise count if data file is available?',
        'Should R5 UNPARSEABLE->majority be disabled unless majority is unanimous, since idx 14 is high-risk?',
        'What is the safest v32 direction if v31 rolls back: new structured generation, symbolic premise graph, or candidate reranking with no trained model?',
    ],
    proposed_next_directions=[
        'v32_candidate_generation_refresh: generate a small new candidate pool for remaining YNU errors with stricter output schema and no training.',
        'v32_symbolic_premise_graph: parse natural-language premise chains into a graph and detect missing links / converse fallacy without overriding MC.',
        'v32_pool_oracle_audit: compute for each wrong case whether any candidate has gold answer; if no, training/generation is needed; if yes, selector is the bottleneck.',
        'v32_yes_no_directional_verifier: separate verifier for No->Yes and Yes->No with asymmetric thresholds to avoid Yes collapse.',
    ],
)
report_path = os.path.join(WORKDIR, 'v31_unified_report.json')
dump_json(v31_unified_report, report_path)
print('\nDONE v31 unified.')
print('Final selection:', v31_full_summary['selection_status'])
print('Final source:', v31_full_summary['selected_source'])
print('Final metrics:', v31_full_summary['metrics'])
print('Report:', report_path)
print('Outputs are in:', WORKDIR)
